# Zomato Dataset Analysis

**Internship Task:** Zomato Dataset Analysis  
**Dataset:** `bhanupratapbiswas/zomato` from Kaggle

Goal: analyze restaurant and review data to extract insights on ratings, cuisines, location preferences, price, and factors affecting ratings.

## 1. Install and Import Libraries

In [ ]:
# Run this cell first in Google Colab or Jupyter.
# If a package is already installed, this is safe.
import sys
!{sys.executable} -m pip install -q pandas numpy matplotlib seaborn wordcloud kaggle

from pathlib import Path
import os
import re
import zipfile
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titleweight'] = 'bold'
pd.set_option('display.max_columns', 50)

## 2. Download the Kaggle Dataset

If running in Google Colab, upload your `kaggle.json` API key when asked. You can create it from Kaggle: Account -> API -> Create New Token.

In [ ]:
DATASET = 'bhanupratapbiswas/zomato'
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

def setup_kaggle_key_if_colab():
    try:
        from google.colab import files
        if not Path('/root/.kaggle/kaggle.json').exists():
            print('Upload kaggle.json now...')
            uploaded = files.upload()
            if 'kaggle.json' in uploaded:
                Path('/root/.kaggle').mkdir(exist_ok=True)
                Path('/root/.kaggle/kaggle.json').write_bytes(uploaded['kaggle.json'])
                os.chmod('/root/.kaggle/kaggle.json', 0o600)
    except Exception:
        pass

if not list(DATA_DIR.glob('*.csv')):
    setup_kaggle_key_if_colab()
    try:
        subprocess.run(['kaggle', 'datasets', 'download', '-d', DATASET, '-p', str(DATA_DIR), '--unzip'], check=True)
    except Exception as exc:
        print('Automatic Kaggle download failed.')
        print('Manual option: download the dataset ZIP from Kaggle and upload/extract zomato.csv into the data folder.')
        raise exc

csv_files = list(DATA_DIR.rglob('*.csv'))
csv_files

## 3. Load and Inspect Data

In [ ]:
csv_path = csv_files[0]
df = pd.read_csv(csv_path, on_bad_lines='skip')

print('Shape:', df.shape)
display(df.head())
display(df.info())

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing)

## 4. Data Cleaning

In [ ]:
clean = df.copy()

# Standardize text columns
for col in clean.select_dtypes(include='object').columns:
    clean[col] = clean[col].astype(str).str.strip()
    clean[col] = clean[col].replace({'nan': np.nan, 'None': np.nan, '': np.nan})

# Convert ratings like '4.1/5' into 4.1. NEW and '-' become NaN.
clean['rating'] = (
    clean['rate']
    .astype(str)
    .str.extract(r'(\\d+(?:\\.\\d+)?)')[0]
    .astype(float)
)
clean.loc[(clean['rating'] < 0) | (clean['rating'] > 5), 'rating'] = np.nan

# Convert cost for two people into numeric rupee amount.
cost_col = 'approx_cost(for two people)'
clean['cost_for_two'] = (
    clean[cost_col]
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.extract(r'(\\d+(?:\\.\\d+)?)')[0]
    .astype(float)
)

clean['votes'] = pd.to_numeric(clean['votes'], errors='coerce')

# Keep valid Yes/No category rows for relationship analysis.
valid = clean[
    clean['rating'].notna()
    & clean['online_order'].isin(['Yes', 'No'])
    & clean['book_table'].isin(['Yes', 'No'])
].copy()

print('Original rows:', len(df))
print('Clean valid rows:', len(valid))
print('Average rating:', round(valid['rating'].mean(), 2))
print('Average cost for two:', round(valid['cost_for_two'].mean(), 0))
display(valid.head())

In [ ]:
cleaning_summary = pd.DataFrame({
    'missing_before': df.isna().sum(),
    'missing_after_basic_cleaning': clean.isna().sum()
}).sort_values('missing_after_basic_cleaning', ascending=False)
display(cleaning_summary)

## 5. Visualizations

In [ ]:
plt.figure(figsize=(11, 5))
sns.heatmap(clean.isna(), cbar=False, yticklabels=False, cmap='mako')
plt.title('Missing Values Heatmap')
plt.xlabel('Columns')
plt.show()

In [ ]:
sns.histplot(valid['rating'], bins=20, kde=True, color='#2a9d8f')
plt.title('Distribution of Restaurant Ratings')
plt.xlabel('Rating')
plt.ylabel('Restaurant Count')
plt.show()

In [ ]:
top_locations = valid['location'].value_counts().head(12)
sns.barplot(x=top_locations.values, y=top_locations.index, palette='viridis')
plt.title('Top Restaurant Location Hotspots')
plt.xlabel('Restaurant Count')
plt.ylabel('Location')
plt.show()

In [ ]:
location_rating = (
    valid.groupby('location')
    .agg(count=('name', 'count'), avg_rating=('rating', 'mean'))
    .query('count >= 100')
    .sort_values('avg_rating', ascending=False)
    .head(12)
)
sns.barplot(data=location_rating, x='avg_rating', y=location_rating.index, palette='crest')
plt.title('Best Rated Locations with at Least 100 Restaurants')
plt.xlabel('Average Rating')
plt.ylabel('Location')
plt.xlim(3.4, 4.3)
plt.show()
display(location_rating)

In [ ]:
cuisine_df = valid[['cuisines', 'rating']].dropna().copy()
cuisine_df['cuisine'] = cuisine_df['cuisines'].str.split(',')
cuisine_df = cuisine_df.explode('cuisine')
cuisine_df['cuisine'] = cuisine_df['cuisine'].str.strip()
cuisine_summary = (
    cuisine_df.groupby('cuisine')
    .agg(count=('rating', 'size'), avg_rating=('rating', 'mean'))
    .query('count >= 100')
)

top_cuisines = cuisine_summary.sort_values('count', ascending=False).head(12)
sns.barplot(data=top_cuisines, x='count', y=top_cuisines.index, palette='rocket')
plt.title('Most Common Cuisines')
plt.xlabel('Count')
plt.ylabel('Cuisine')
plt.show()
display(top_cuisines)

In [ ]:
best_cuisines = cuisine_summary.sort_values('avg_rating', ascending=False).head(12)
sns.barplot(data=best_cuisines, x='avg_rating', y=best_cuisines.index, palette='flare')
plt.title('Highest Rated Cuisines with at Least 100 Restaurants')
plt.xlabel('Average Rating')
plt.ylabel('Cuisine')
plt.xlim(3.8, 4.4)
plt.show()
display(best_cuisines)

In [ ]:
valid['price_segment'] = pd.cut(
    valid['cost_for_two'],
    bins=[0, 300, 600, 1000, 2000, np.inf],
    labels=['Budget <=300', 'Value 301-600', 'Mid 601-1000', 'Premium 1001-2000', 'Luxury >2000']
)
price_rating = valid.groupby('price_segment', observed=False).agg(
    count=('name', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()

sns.barplot(data=price_rating, x='price_segment', y='avg_rating', palette='magma')
plt.title('Price Segment vs Average Rating')
plt.xlabel('Price Segment')
plt.ylabel('Average Rating')
plt.xticks(rotation=25, ha='right')
plt.ylim(3.3, 4.3)
plt.show()
display(price_rating)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=valid, x='online_order', y='rating', ax=axes[0], palette='Set2')
axes[0].set_title('Online Order vs Rating')
axes[0].set_xlabel('Online Order')
axes[0].set_ylabel('Average Rating')
axes[0].set_ylim(3.4, 4.3)

sns.barplot(data=valid, x='book_table', y='rating', ax=axes[1], palette='Set3')
axes[1].set_title('Table Booking vs Rating')
axes[1].set_xlabel('Book Table')
axes[1].set_ylabel('Average Rating')
axes[1].set_ylim(3.4, 4.3)
plt.tight_layout()
plt.show()

display(valid.groupby('online_order')['rating'].agg(['count', 'mean']))
display(valid.groupby('book_table')['rating'].agg(['count', 'mean']))

In [ ]:
corr = valid[['rating', 'cost_for_two', 'votes']].corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Heatmap')
plt.show()
display(corr)

In [ ]:
dish_text = ' '.join(valid['dish_liked'].dropna().astype(str))
wordcloud = WordCloud(width=1100, height=500, background_color='white', colormap='tab10').generate(dish_text)
plt.figure(figsize=(13, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Popular Dishes')
plt.show()

## 6. Key Findings

In [ ]:
summary = {
    'Total rows': len(df),
    'Clean valid rows': len(valid),
    'Average rating': round(valid['rating'].mean(), 2),
    'Average cost for two': round(valid['cost_for_two'].mean(), 0),
    'Price-rating correlation': round(valid[['cost_for_two', 'rating']].dropna().corr().iloc[0, 1], 3),
    'Votes-rating correlation': round(valid[['votes', 'rating']].dropna().corr().iloc[0, 1], 3),
}
display(pd.Series(summary, name='Value'))

print('Top location by restaurant count:', valid['location'].value_counts().idxmax())
print('Highest average rated location among locations with 100+ restaurants:', location_rating.index[0])
print('Most common cuisine:', top_cuisines.index[0])
print('Highest average rated cuisine among cuisines with 100+ restaurants:', best_cuisines.index[0])

## 7. Recommendations for Alfido Tech

1. **Focus restaurant discovery on high-performing localities.** Promote premium food hubs like Lavelle Road, St. Marks Road, Church Street, and Koramangala because they show consistently strong ratings.

2. **Use table booking as a quality signal.** Restaurants with table booking have much higher ratings than restaurants without table booking, so this feature should receive weight in recommendations.

3. **Segment restaurants by price.** Premium restaurants usually have higher ratings, while budget restaurants have high volume. Separate budget, mid-range, and premium rankings will produce fairer recommendations.

4. **Promote niche high-rated cuisines.** Modern Indian, Japanese, Mediterranean, European, Korean, and Asian cuisines perform strongly and can be highlighted for users looking for quality dining.

5. **Use popular dishes for tagging and campaigns.** Pasta, burgers, cocktails, pizza, biryani, and coffee appear frequently in liked dishes, so they can improve search, filters, and promotional content.

## 8. Conclusion

The analysis shows that ratings are affected by multiple factors including location, cuisine, price, table booking, and customer engagement. Table booking and premium dining segments are especially strong indicators of higher ratings. A practical recommendation system should combine location, rating, cost segment, cuisine type, table booking availability, and popular dish tags.